In [1]:
# Import necessary libraries
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import numpy as np

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
import requests
from shapely.geometry import Point

# Overpass query for Spätis & corner stores in Berlin
query = """
[out:json];
(
  node[shop=convenience](52.4,13.2,52.7,13.6);
  node[shop=kiosk](52.4,13.2,52.7,13.6);
);
out body;
"""

# Send request
url = "https://overpass-api.de/api/interpreter"
resp = requests.post(url, data=query, timeout=180)
resp.raise_for_status()
data = resp.json()

# Normalize JSON to pandas dataframe
elements = data.get("elements", [])
df = pd.json_normalize(elements)

# Ensure lat/lon exist
df = df[df['lat'].notna() & df['lon'].notna()]

# Convert to GeoDataFrame
spatis_gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df['lon'].astype(float), df['lat'].astype(float))],
    crs="EPSG:4326"
)

# Minimal cleanup and renaming to match schema
spatis = spatis_gdf.rename(columns={
    'tags.name': 'name',
    'geometry': 'geometry', 
    'tags.brand': 'brand',
    'tags.addr:street' : 'address',
    'tags.operator': 'operator',
    'tags.opening_hours': 'opening_hours',
    'tags.phone': 'phone_number',
    'tags.website': 'website',
    'tags.email': 'email',
    'tags.source': 'source',
    'lat': 'latitude',
    'lon': 'longitude',
    'type': 'osm_type',
    'id': 'spaeti_id'  # Keep OSM ID as 'id'
}).copy()

print("SPATIS fetched and loaded. Records:", len(spatis))
spatis.head(5)


SPATIS fetched and loaded. Records: 1608


,osm_type,spaeti_id,latitude,longitude,tags.addr:city,tags.addr:country,tags.addr:housenumber,tags.addr:postcode,address,tags.addr:suburb,...,tags.post_office:opening_hours,tags.money_transfer,tags.mobile,tags.fuel:HGV_diesel,tags.fuel:octane_100,tags.post_office:id_check,tags.name:ko,tags.public_transport,tags.ticket,geometry
0,node,26867411,52.501974,13.294496,Berlin,DE,14,10711,Heilbronner Straße,Halensee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.2945 52.50197)
1,node,29997723,52.508370,13.280947,Berlin,DE,8-10,14057,Messedamm,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.28095 52.50837)
2,node,63253672,52.499322,13.296118,Berlin,DE,39,10711,Joachim-Friedrich-Straße,Halensee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.29612 52.49932)
3,node,253616592,52.511047,13.462698,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.4627 52.51105)
4,node,266629404,52.509209,13.587500,Berlin,DE,45,12621,Mädewalder Weg,Kaulsdorf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.5875 52.50921)


In [ ]:
print("Columns in spatis:", spatis.columns.tolist())

In [4]:
# Explore all columns and get summary statistics
spatis.describe(include="all")


,osm_type,spaeti_id,latitude,longitude,tags.addr:city,tags.addr:country,tags.addr:housenumber,tags.addr:postcode,address,tags.addr:suburb,...,tags.post_office:opening_hours,tags.money_transfer,tags.mobile,tags.fuel:HGV_diesel,tags.fuel:octane_100,tags.post_office:id_check,tags.name:ko,tags.public_transport,tags.ticket,geometry
count,1608,1.608000e+03,1608.000000,1608.000000,687,490,797,708,831,476,...,2,1,2,1,1,1,1,1,1,1608
unique,1,NaN,NaN,NaN,6,1,233,149,439,53,...,2,1,2,1,1,1,1,1,1,1606
top,node,NaN,NaN,NaN,Berlin,DE,1,10245,Karl-Marx-Straße,Prenzlauer Berg,...,"Fr 07:00-19:00, Sa 09:00-14:00",Western Union;ria,+491784027368,yes,yes,yes,삼일상사,service_center,public_transport,POINT (13.3129938 52.5017546)
freq,1608,NaN,NaN,NaN,678,490,19,34,14,60,...,1,1,1,1,1,1,1,1,1,2
mean,NaN,5.371704e+09,52.512158,13.396506,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,3.784769e+09,0.040225,0.068706,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,2.686741e+07,52.401392,13.200417,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,1.987385e+09,52.488111,13.348118,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,4.522934e+09,52.510001,13.402172,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,8.331316e+09,52.540391,13.439918,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Check missing values in each column
missing_count = spatis.isna().sum().sort_values(ascending=False)

# List columns where missing values are greater than 200
print(missing_count[missing_count > 200])


tags.sells:tobacco                1607
tags.note:de                      1607
tags.reusable_packaging:accept    1607
tags.wikidata                     1607
tags.wikipedia                    1607
                                  ... 
tags.addr:postcode                 900
opening_hours                      888
tags.addr:housenumber              811
address                            777
tags.wheelchair                    605
Length: 245, dtype: int64


In [6]:
# Check unique values in the 'brand' column
spatis['brand'].value_counts()



brand
REWE To Go                                  16
ServiceStore DB                             15
DHL                                          4
Yorma's                                      2
Total                                        2
Spar                                         2
DPD                                          1
JET                                          1
Shell Shop                                   1
Shell                                        1
Weltladen                                    1
Aral                                         1
Deutsche Post                                1
EDEKA                                        1
Elan                                         1
Lycamobile                                   1
Deutsche Post;DHL;Postbank;Western Union     1
Agip                                         1
Hermes                                       1
Edeka                                        1
TotalEnergies                                1
Name: c

In [7]:
# expand all columns to see more details
pd.set_option('display.max_columns', None)

print(spatis_gdf.head(3))


   type        id        lat        lon tags.addr:city tags.addr:country  \
0  node  26867411  52.501974  13.294496         Berlin                DE   
1  node  29997723  52.508370  13.280947         Berlin                DE   
2  node  63253672  52.499322  13.296118         Berlin                DE   

  tags.addr:housenumber tags.addr:postcode          tags.addr:street  \
0                    14              10711        Heilbronner Straße   
1                  8-10              14057                 Messedamm   
2                    39              10711  Joachim-Friedrich-Straße   

  tags.addr:suburb tags.amenity tags.check_date:opening_hours  \
0         Halensee         fuel                    2022-04-01   
1              NaN         fuel                           NaN   
2         Halensee          NaN                           NaN   

  tags.compressed_air tags.fuel:adblue tags.fuel:biodiesel tags.fuel:diesel  \
0                 yes              yes                 yes        

In [8]:
# Modelling & Planning
# Export SPATIS GeoDataFrame to the Downloads/Mapping folder

# Export as GeoJSON
spatis_gdf.to_file(
    "/Users/harrisongoodman/Downloads/spatis_raw.geojson",
    driver="GeoJSON"
)

# Export as CSV without geometry
spatis_gdf.drop(columns="geometry").to_csv(
    "/Users/harrisongoodman/Downloads/spatis_raw.csv",
    index=False
)

print("Raw SPATIS exported. Records:", len(spatis_gdf))


Raw SPATIS exported. Records: 1608


In [9]:
# Load districts
districts = gpd.read_file("/Users/harrisongoodman/Downloads/bezirksgrenzen.geojson")

# Rename correctly
districts = districts.rename(columns={
    "Gemeinde_schluessel": "district_id",
    "Gemeinde_name": "district_name"
})

# Keep only the necessary columns
districts = districts[["district_id", "district_name", "geometry"]].copy()

# Convert types
districts["district_id"] = districts["district_id"].astype(str)

print(districts.head())





  district_id               district_name  \
0         012               Reinickendorf   
1         004  Charlottenburg-Wilmersdorf   
2         009            Treptow-Köpenick   
3         003                      Pankow   
4         008                    Neukölln   

                                            geometry  
0  MULTIPOLYGON (((13.32074 52.6266, 13.32045 52....  
1  MULTIPOLYGON (((13.32111 52.52446, 13.32103 52...  
2  MULTIPOLYGON (((13.57925 52.39083, 13.57958 52...  
3  MULTIPOLYGON (((13.50481 52.6196, 13.50467 52....  
4  MULTIPOLYGON (((13.45832 52.48569, 13.45823 52...  


In [10]:
# Spatial join: assign each SPATIS point to a district
spatis = gpd.sjoin(
    spatis,
    districts[["district_id", "district_name", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")



In [11]:
# Load neighborhoods
neighborhoods = gpd.read_file("/Users/harrisongoodman/Downloads/lor_ortsteile.geojson")

# Correct: rename first
neighborhoods = neighborhoods.rename(columns={
    "spatial_name": "neighborhood_id",      # ID
    "OTEIL": "neighborhood_name"            # name
})

# Now select the correct columns
neighborhoods = neighborhoods[["neighborhood_id", "neighborhood_name", "geometry"]].copy()

# Convert ID to string
neighborhoods["neighborhood_id"] = neighborhoods["neighborhood_id"].astype(str)

# CRS fix to match SPATIS
neighborhoods = neighborhoods.to_crs(spatis.crs)

print(neighborhoods.head())

  neighborhood_id neighborhood_name  \
0            0101             Mitte   
1            0102            Moabit   
2            0103      Hansaviertel   
3            0104        Tiergarten   
4            0105           Wedding   

                                            geometry  
0  POLYGON ((13.41649 52.52696, 13.41635 52.52702...  
1  POLYGON ((13.33884 52.51974, 13.33884 52.51974...  
2  POLYGON ((13.34322 52.51557, 13.34323 52.51557...  
3  POLYGON ((13.36879 52.49878, 13.36891 52.49877...  
4  POLYGON ((13.34656 52.53879, 13.34664 52.53878...  


In [12]:

# Spatial join: neighborhoods

spatis = gpd.sjoin(
    spatis,
    neighborhoods[["neighborhood_id", "neighborhood_name", "geometry"]],
    how="left",
    predicate="within"
)

# Drop the index_right column added by sjoin
spatis = spatis.drop(columns=["index_right"], errors="ignore")

# Keep only the right-hand join columns and rename them
for col in ["neighborhood_id", "neighborhood_name"]:
    if f"{col}_right" in spatis.columns:
        spatis[col] = spatis[f"{col}_right"]

# Drop any leftover left/right duplicate columns
spatis = spatis.drop(columns=[
    "neighborhood_id_left", "neighborhood_id_right",
    "neighborhood_name_left", "neighborhood_name_right"
], errors="ignore")



In [13]:

if "district_id_right" in spatis.columns:
    spatis["district_id"] = spatis["district_id_right"]
    spatis["district_name"] = spatis["district_name_right"]
    spatis = spatis.drop(columns=[
        "district_id_left", "district_id_right",
        "district_name_left", "district_name_right"
    ], errors="ignore")



In [14]:

# --- District mapping (official Berlin codes as strings) ---
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Apply mapping to create district_id
spatis['district_id'] = spatis['district_name'].map(district_mapping)

# (Optional) Check unmapped
unmapped = spatis[spatis['district_id'].isna()]['district_name'].unique()
if len(unmapped) > 0:
    print("⚠️ Unmapped districts:", unmapped)
else:
    print("✓ All districts mapped")




⚠️ Unmapped districts: [nan]


In [15]:
spatis = spatis.drop_duplicates(subset=["spaeti_id"])

print("Duplicates removed. Records remaining:", len(spatis))


Duplicates removed. Records remaining: 1608


In [16]:

spatis = spatis.replace({None: np.nan})


In [17]:

print(spatis.columns.tolist())
print(spatis.head(3))


['osm_type', 'spaeti_id', 'latitude', 'longitude', 'tags.addr:city', 'tags.addr:country', 'tags.addr:housenumber', 'tags.addr:postcode', 'address', 'tags.addr:suburb', 'tags.amenity', 'tags.check_date:opening_hours', 'tags.compressed_air', 'tags.fuel:adblue', 'tags.fuel:biodiesel', 'tags.fuel:diesel', 'tags.fuel:e10', 'tags.fuel:octane_95', 'tags.fuel:octane_98', 'name', 'opening_hours', 'operator', 'tags.shop', 'tags.wheelchair', 'brand', 'tags.brand:wikidata', 'tags.brand:wikipedia', 'tags.fuel:GTL_diesel', 'tags.fuel:biogas', 'tags.fuel:cng', 'tags.fuel:lpg', 'tags.fuel:octane_102', 'tags.surveillance', 'website', 'tags.check_date', 'tags.dog', 'email', 'tags.fax', 'phone_number', 'tags.start_date', 'tags.indoor_seating', 'tags.organic', 'tags.outdoor_seating', 'tags.smoking', 'tags.opening_hours:signed', 'tags.diet:halal', 'tags.level', 'tags.payment:credit_cards', 'tags.payment:debit_cards', 'tags.payment:apple_pay', 'tags.payment:cards', 'tags.payment:cash', 'tags.payment:google_

In [18]:
df = df.rename(columns={"id": "spaeti_id"})


In [19]:
selected_columns = [
    "spaeti_id",
    "name",
    "brand",
    "operator",
    "latitude",
    "longitude",
    "district_id",
    "district_name",
    "neighborhood_id",
    "neighborhood_name",
    "opening_hours",  
    "phone_number",
    "website",
    "email",
    "geometry",
    "address"
]



In [20]:
final_cols = [
    "spaeti_id", "name", "brand", "operator",
    "latitude", "longitude",
    "district_id", "district_name",
    "neighborhood_id", "neighborhood_name",
    "opening_hours",   # <-- correct name after renaming
    "phone_number", "website","email", "geometry",
   
    "address"
]





In [21]:
# Define the columns you want in the final output
final_cols = [
    "spaeti_id", "name", "brand", "operator",
    "latitude", "longitude",
    "district_id", "district_name",
    "neighborhood_id", "neighborhood_name",
    "opening_hours", "phone_number", "website","email",
    "address", "geometry",
]

# Keep only columns that exist in the joined spatis
final_cols = [c for c in final_cols if c in spatis.columns]

# Remove duplicate IDs
spatis = spatis.drop_duplicates(subset=["spaeti_id"])

# Select the final columns
spatis = spatis[final_cols]

# Check result
spatis.head(5)








,spaeti_id,name,brand,operator,latitude,longitude,district_id,district_name,neighborhood_id,neighborhood_name,opening_hours,phone_number,website,email,address,geometry
0,26867411,Bavaria petrol,NaN,Bavaria Petrol,52.501974,13.294496,11004004,Charlottenburg-Wilmersdorf,0407,Halensee,Mo-Fr 07:00-22:00; Sa 08:00-22:00,NaN,NaN,NaN,Heilbronner Straße,POINT (13.2945 52.50197)
1,29997723,Aral,Aral,Anne Notzke,52.508370,13.280947,11004004,Charlottenburg-Wilmersdorf,0405,Westend,24/7,NaN,https://tankstelle.aral.de/tankstelle/berlin/m...,NaN,Messedamm,POINT (13.28095 52.50837)
2,63253672,Späti Joe,NaN,NaN,52.499322,13.296118,11004004,Charlottenburg-Wilmersdorf,0407,Halensee,24/7,NaN,NaN,NaN,Joachim-Friedrich-Straße,POINT (13.29612 52.49932)
3,253616592,Mein Markt Pham,NaN,NaN,52.511047,13.462698,11002002,Friedrichshain-Kreuzberg,0201,Friedrichshain,Mo-Fr 08:00-20:00; Sa 08:00-19:00,NaN,NaN,NaN,NaN,POINT (13.4627 52.51105)
4,266629404,...nah und gut,EDEKA,Heinz Vollack,52.509209,13.587500,11010010,Marzahn-Hellersdorf,1003,Kaulsdorf,Mo-Fr 07:00-19:00; Sa 07:00-13:00; PH off,+49 30 5677706,NaN,e-markt-vollack@t-online.de,Mädewalder Weg,POINT (13.5875 52.50921)


In [22]:
# Count missing district_id and district_name
print("Missing district_id:", spatis['district_id'].isna().sum())
print("Missing district_name:", spatis['district_name'].isna().sum())

Missing district_id: 25
Missing district_name: 25


In [23]:
spatis = spatis.dropna(subset=['district_id', 'district_name'])
print("After dropping missing districts, rows left:", len(spatis))

After dropping missing districts, rows left: 1583


In [24]:
import os

OUTPUT_DIR = "/Users/harrisongoodman/Downloads/"

# Ensure path ends with /
if not OUTPUT_DIR.endswith("/"):
    OUTPUT_DIR += "/"

# --- Export the final spatis dataframe ---
spatis.to_csv(
    os.path.join(OUTPUT_DIR, "spatis_with_admins.csv"),
    index=False
)

print("✅ Exported: spatis_with_admins.csv → Downloads")


✅ Exported: spatis_with_admins.csv → Downloads


In [25]:
print("🔎 Preview of data being exported:")
display(spatis.head(10))     # first 10 rows
display(spatis.sample(5))    # 5 random rows
print("\n📐 DataFrame info:")
print(spatis.info())


🔎 Preview of data being exported:


,spaeti_id,name,brand,operator,latitude,longitude,district_id,district_name,neighborhood_id,neighborhood_name,opening_hours,phone_number,website,email,address,geometry
0,26867411,Bavaria petrol,NaN,Bavaria Petrol,52.501974,13.294496,11004004,Charlottenburg-Wilmersdorf,0407,Halensee,Mo-Fr 07:00-22:00; Sa 08:00-22:00,NaN,NaN,NaN,Heilbronner Straße,POINT (13.2945 52.50197)
1,29997723,Aral,Aral,Anne Notzke,52.508370,13.280947,11004004,Charlottenburg-Wilmersdorf,0405,Westend,24/7,NaN,https://tankstelle.aral.de/tankstelle/berlin/m...,NaN,Messedamm,POINT (13.28095 52.50837)
2,63253672,Späti Joe,NaN,NaN,52.499322,13.296118,11004004,Charlottenburg-Wilmersdorf,0407,Halensee,24/7,NaN,NaN,NaN,Joachim-Friedrich-Straße,POINT (13.29612 52.49932)
3,253616592,Mein Markt Pham,NaN,NaN,52.511047,13.462698,11002002,Friedrichshain-Kreuzberg,0201,Friedrichshain,Mo-Fr 08:00-20:00; Sa 08:00-19:00,NaN,NaN,NaN,NaN,POINT (13.4627 52.51105)
4,266629404,...nah und gut,EDEKA,Heinz Vollack,52.509209,13.587500,11010010,Marzahn-Hellersdorf,1003,Kaulsdorf,Mo-Fr 07:00-19:00; Sa 07:00-13:00; PH off,+49 30 5677706,NaN,e-markt-vollack@t-online.de,Mädewalder Weg,POINT (13.5875 52.50921)
5,268424318,NaN,NaN,NaN,52.596622,13.290950,11012012,Reinickendorf,1202,Tegel,NaN,NaN,NaN,NaN,NaN,POINT (13.29095 52.59662)
6,269475044,Milchladen,NaN,NaN,52.501786,13.416537,11002002,Friedrichshain-Kreuzberg,0202,Kreuzberg,Mo-Fr 11:00-18:00; Su 13:00-18:00,NaN,NaN,NaN,Dresdener Straße,POINT (13.41654 52.50179)
7,270719151,Zeitungskiosk Bagci,NaN,NaN,52.434623,13.342479,11006006,Steglitz-Zehlendorf,0603,Lankwitz,NaN,NaN,NaN,NaN,Dillgesstraße,POINT (13.34248 52.43462)
8,271419848,Berlin Tag und Nacht 2,NaN,NaN,52.528971,13.315925,11001001,Mitte,0102,Moabit,NaN,NaN,NaN,NaN,NaN,POINT (13.31592 52.52897)
9,277681454,Bagdad Lebensmittel,NaN,NaN,52.499094,13.393187,11002002,Friedrichshain-Kreuzberg,0202,Kreuzberg,NaN,NaN,NaN,NaN,NaN,POINT (13.39319 52.49909)


,spaeti_id,name,brand,operator,latitude,longitude,district_id,district_name,neighborhood_id,neighborhood_name,opening_hours,phone_number,website,email,address,geometry
641,3677450357,Spätie,NaN,NaN,52.562068,13.208788,11005005,Spandau,0507,Hakenfelde,NaN,NaN,NaN,NaN,NaN,POINT (13.20879 52.56207)
995,6192424371,Kiosk,NaN,NaN,52.486011,13.361977,11007007,Tempelhof-Schöneberg,0701,Schöneberg,NaN,NaN,NaN,NaN,NaN,POINT (13.36198 52.48601)
946,5613723540,Viktoria Späti,NaN,NaN,52.488797,13.376466,11002002,Friedrichshain-Kreuzberg,0202,Kreuzberg,NaN,NaN,NaN,NaN,Katzbachstraße,POINT (13.37647 52.4888)
1166,7804824563,Lädchen im sunpark,NaN,NaN,52.463914,13.425674,11008008,Neukölln,0801,Neukölln,Mo-Fr 08:00-14:00; Sa 08:00-12:00,NaN,NaN,NaN,Mariendorfer Weg,POINT (13.42567 52.46391)
1200,8290426023,Büdchen,NaN,NaN,52.502121,13.429036,11002002,Friedrichshain-Kreuzberg,0202,Kreuzberg,NaN,NaN,NaN,NaN,NaN,POINT (13.42904 52.50212)



📐 DataFrame info:
<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 1583 entries, 0 to 1607
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   spaeti_id          1583 non-null   int64   
 1   name               1427 non-null   object  
 2   brand              54 non-null     object  
 3   operator           57 non-null     object  
 4   latitude           1583 non-null   float64 
 5   longitude          1583 non-null   float64 
 6   district_id        1583 non-null   object  
 7   district_name      1583 non-null   object  
 8   neighborhood_id    1583 non-null   object  
 9   neighborhood_name  1583 non-null   object  
 10  opening_hours      708 non-null    object  
 11  phone_number       68 non-null     object  
 12  website            55 non-null     object  
 13  email              15 non-null     object  
 14  address            822 non-null    object  
 15  geometry           1583 non-null 

In [26]:
df.to_csv("spaetis_for_sql_final.csv", index=False)

In [27]:
# Show all columns and their data types
spatis.dtypes.to_frame("dtype")


,dtype
spaeti_id,int64
name,object
brand,object
operator,object
latitude,float64
longitude,float64
district_id,object
district_name,object
neighborhood_id,object
neighborhood_name,object


In [28]:
print(spatis.columns)



Index(['spaeti_id', 'name', 'brand', 'operator', 'latitude', 'longitude',
       'district_id', 'district_name', 'neighborhood_id', 'neighborhood_name',
       'opening_hours', 'phone_number', 'website', 'email', 'address',
       'geometry'],
      dtype='object')


In [29]:
# Next Step: Step 2 — Fetch & Transform data.

In [57]:
import pandas as pd

# Create a sample dataframe instead of using undefined spaetis_final
# Replace this with your actual data loading code
df = pd.DataFrame({
    'spaeti_id': [1, 2, 3, 4, 5],
    'name': ['Späti 1', 'Späti 2', 'Späti 3', 'Späti 4', None],
    'address': ['Address 1', 'Address 2', 'Address 3', None, 'Address 5'],
    'district_id': [1, 2, None, 4, 5],
    'neighborhood_id': [10, 20, 30, None, 50],
    'latitude': [52.5, 52.4, 52.6, 52.8, 52.5],  # Note: one invalid coordinate
    'longitude': [13.4, 13.5, 13.2, 13.3, 14.0],  # Note: one invalid coordinate
    'opening_hours': ['09:00-22:00', '08:00-23:00', 'Always Open', '10:00-20:00', None]
})

# 1. Check for duplicate spaeti_id
duplicates = df['spaeti_id'].duplicated().sum()
print("Duplicate spaeti_id:", duplicates)

# 2. Validate district_id & neighborhood_id mapping
invalid_districts = df[df['district_id'].isna()]
invalid_neighborhoods = df[df['neighborhood_id'].isna()]
print("Missing district_id:", len(invalid_districts))
print("Missing neighborhood_id:", len(invalid_neighborhoods))

# 3. Check coordinates within Berlin
invalid_coords = df[
    ~(
        df['latitude'].between(52.3, 52.7) &
        df['longitude'].between(13.0, 13.8)
    )
]
print("Invalid coordinates:", len(invalid_coords))

# 4. Ensure required fields are populated
required_fields = ['spaeti_id', 'name', 'address']
for field in required_fields:
    missing = df[field].isna().sum()
    print(f"Missing {field}: {missing}")

# 5. Check opening_hours format (basic sanity check)
invalid_hours = df[~df['opening_hours'].str.contains(r'\d{2}:\d{2}', na=False)]
print("Rows with invalid opening_hours:", len(invalid_hours))

Duplicate spaeti_id: 0
Missing district_id: 1
Missing neighborhood_id: 1
Invalid coordinates: 2
Missing spaeti_id: 0
Missing name: 1
Missing address: 1
Rows with invalid opening_hours: 2


In [61]:
import pandas as pd
import os

# Use your existing dataframe with spaeti_id
df = spatis  # or spaetis_final, whichever variable you have


# 1️⃣ Remove duplicate spaeti_id

df = df.drop_duplicates(subset=["spaeti_id"])


# 2️⃣ Create "coordinates" from geometry ONLY if geometry exists

if "geometry" in df.columns:
    df["coordinates"] = df["geometry"].astype(str)
else:
    if "coordinates" not in df.columns:
        df["coordinates"] = pd.NA  # placeholder for SQL


# 3️⃣ Define SQL-required column order

sql_columns = [
    "spaeti_id", "district_id", "neighborhood_id", "name", "type",
    "address", "postal_code", "phone_number", "email", "website",
    "coordinates", "latitude", "longitude", "opening_hours",
    "beverage_emphasis", "has_outdoor_area",
    "district_name", "neighborhood_name","geometry"
]

# Keep only columns that exist in df
sql_columns_existing = [c for c in sql_columns if c in df.columns]


# 4️⃣ Reorder columns safely

df_export = df[sql_columns_existing]


# 5️⃣ Preview data before export

print("🔎 Preview of final export (10 rows):")
display(df_export.head(10))

print("\n📐 DataFrame info:")
print(df_export.info())

print("\n📊 Columns in final export:")
print(list(df_export.columns))


# 6️⃣ Export CSV (without geometry)

OUTPUT_DIR = "/Users/harrisongoodman/Downloads/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

export_path = os.path.join(OUTPUT_DIR, "spaetis_for_sql_final.csv")

# Drop geometry if exists
if "geometry" in df_export.columns:
    df_export = df_export.drop(columns=["geometry"])

df_export.to_csv(export_path, index=False)
print(f"\n✅ Exported CSV to: {export_path}")


🔎 Preview of final export (10 rows):


,spaeti_id,district_id,neighborhood_id,name,address,phone_number,email,website,coordinates,latitude,longitude,opening_hours,district_name,neighborhood_name,geometry
0,26867411,11004004,0407,Bavaria petrol,Heilbronner Straße,NaN,NaN,NaN,POINT (13.294496 52.501974),52.501974,13.294496,Mo-Fr 07:00-22:00; Sa 08:00-22:00,Charlottenburg-Wilmersdorf,Halensee,POINT (13.2945 52.50197)
1,29997723,11004004,0405,Aral,Messedamm,NaN,NaN,https://tankstelle.aral.de/tankstelle/berlin/m...,POINT (13.280947 52.50837),52.508370,13.280947,24/7,Charlottenburg-Wilmersdorf,Westend,POINT (13.28095 52.50837)
2,63253672,11004004,0407,Späti Joe,Joachim-Friedrich-Straße,NaN,NaN,NaN,POINT (13.296118 52.499322),52.499322,13.296118,24/7,Charlottenburg-Wilmersdorf,Halensee,POINT (13.29612 52.49932)
3,253616592,11002002,0201,Mein Markt Pham,NaN,NaN,NaN,NaN,POINT (13.462698 52.511047),52.511047,13.462698,Mo-Fr 08:00-20:00; Sa 08:00-19:00,Friedrichshain-Kreuzberg,Friedrichshain,POINT (13.4627 52.51105)
4,266629404,11010010,1003,...nah und gut,Mädewalder Weg,+49 30 5677706,e-markt-vollack@t-online.de,NaN,POINT (13.5875 52.509209),52.509209,13.587500,Mo-Fr 07:00-19:00; Sa 07:00-13:00; PH off,Marzahn-Hellersdorf,Kaulsdorf,POINT (13.5875 52.50921)
5,268424318,11012012,1202,NaN,NaN,NaN,NaN,NaN,POINT (13.29095 52.596622),52.596622,13.290950,NaN,Reinickendorf,Tegel,POINT (13.29095 52.59662)
6,269475044,11002002,0202,Milchladen,Dresdener Straße,NaN,NaN,NaN,POINT (13.416537 52.501786),52.501786,13.416537,Mo-Fr 11:00-18:00; Su 13:00-18:00,Friedrichshain-Kreuzberg,Kreuzberg,POINT (13.41654 52.50179)
7,270719151,11006006,0603,Zeitungskiosk Bagci,Dillgesstraße,NaN,NaN,NaN,POINT (13.342479 52.434623),52.434623,13.342479,NaN,Steglitz-Zehlendorf,Lankwitz,POINT (13.34248 52.43462)
8,271419848,11001001,0102,Berlin Tag und Nacht 2,NaN,NaN,NaN,NaN,POINT (13.315925 52.528971),52.528971,13.315925,NaN,Mitte,Moabit,POINT (13.31592 52.52897)
9,277681454,11002002,0202,Bagdad Lebensmittel,NaN,NaN,NaN,NaN,POINT (13.393187 52.499094),52.499094,13.393187,NaN,Friedrichshain-Kreuzberg,Kreuzberg,POINT (13.39319 52.49909)



📐 DataFrame info:
<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 1583 entries, 0 to 1607
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   spaeti_id          1583 non-null   int64   
 1   district_id        1583 non-null   object  
 2   neighborhood_id    1583 non-null   object  
 3   name               1427 non-null   object  
 4   address            822 non-null    object  
 5   phone_number       68 non-null     object  
 6   email              15 non-null     object  
 7   website            55 non-null     object  
 8   coordinates        1583 non-null   object  
 9   latitude           1583 non-null   float64 
 10  longitude          1583 non-null   float64 
 11  opening_hours      708 non-null    object  
 12  district_name      1583 non-null   object  
 13  neighborhood_name  1583 non-null   object  
 14  geometry           1583 non-null   geometry
dtypes: float64(2), geometry(1), int64

In [642]:
print(spaetis_final.columns.tolist())


['spaeti_id', 'district_id', 'neighborhood_id', 'name', 'address', 'postal_code', 'phone_number', 'email', 'website', 'opening_hours', 'beverage_emphasis', 'has_outdoor_area', 'district_name', 'neighborhood_name', 'latitude', 'longitude', 'geometry', 'type', 'coordinates']


In [8]:
import pandas as pd
import os

# Use your current dataframe
df = spatis  # or spaetis_final


#  Quick preview
print("🔎 Preview of input dataframe (5 rows):")
display(df.head())

#  Remove duplicate spaeti_id
if "spaeti_id" not in df.columns:
    raise KeyError("❌ 'spaeti_id' column not found!")
df = df.drop_duplicates(subset=["spaeti_id"])


# Ensure optional boolean columns exist
for col in ["beverage_emphasis", "has_outdoor_area"]:
    if col not in df.columns:
        df[col] = False  # default value



#  Data Quality Checks (EPIC 2)
print("\n=== DATA QUALITY CHECKS ===")

# Required fields
required_cols = ["name", "address", "district_id"]
for col in required_cols:
    if col in df.columns:
        print(f"Missing {col}: {df[col].isna().sum()}")
    else:
        print(f" Column '{col}' not found!")

# Duplicate spaeti_id
print("Duplicate spaeti_id:", df["spaeti_id"].duplicated().sum())

# Coordinates validation (Berlin bounding box)
if "latitude" in df.columns and "longitude" in df.columns:
    invalid_coords = df[~((df["latitude"].between(52.3, 52.7)) & 
                          (df["longitude"].between(13.0, 13.8)))]
    print("Invalid coordinates:", len(invalid_coords))
else:
    print(" Latitude/Longitude columns not found!")

# Neighborhood completeness
if "neighborhood_id" in df.columns:
    print("Rows missing neighborhood_id:", df["neighborhood_id"].isna().sum())
else:
    print(" Column 'neighborhood_id' not found!")

# Optional metadata completeness
optional_cols = ["phone_number", "website", "opening_hours"]
for col in optional_cols:
    if col in df.columns:
        print(f"Missing {col}:", df[col].isna().sum())
    else:
        print(f" Column '{col}' not found!")


# 5️⃣ Define SQL-required column order (geometry kept)
sql_columns = [
    "spaeti_id", "district_id", "neighborhood_id", "name", "type",
    "address", "postal_code", "phone_number", "email", "website",
    "latitude", "longitude", "opening_hours",
    "beverage_emphasis", "has_outdoor_area",
    "district_name", "neighborhood_name", "geometry"
]

# Keep only columns that exist in df
sql_columns_existing = [c for c in sql_columns if c in df.columns]


# 6️⃣ Reorder dataframe safely
df_export = df[sql_columns_existing]

# 🚫 DO NOT drop geometry anymore


# Preview final export
print("\n🔎 Preview of final export (10 rows):")
display(df_export.head(10))

print("\n📐 DataFrame info:")
df_export.info()

print("\n📊 Columns in final export:")
print(list(df_export.columns))


#  Export CSV for Postgres ingestion
OUTPUT_DIR = "/Users/harrisongoodman/Downloads/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
export_path = os.path.join(OUTPUT_DIR, "spaetis_for_sql_final.csv")

df_export.to_csv(export_path, index=False)
print(f"\n✅ Exported CSV to: {export_path}")


🔎 Preview of input dataframe (5 rows):


,osm_type,spaeti_id,latitude,longitude,tags.addr:city,tags.addr:country,tags.addr:housenumber,tags.addr:postcode,address,tags.addr:suburb,...,tags.post_office:opening_hours,tags.money_transfer,tags.mobile,tags.fuel:HGV_diesel,tags.fuel:octane_100,tags.post_office:id_check,tags.name:ko,tags.public_transport,tags.ticket,geometry
0,node,26867411,52.501974,13.294496,Berlin,DE,14,10711,Heilbronner Straße,Halensee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.2945 52.50197)
1,node,29997723,52.508370,13.280947,Berlin,DE,8-10,14057,Messedamm,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.28095 52.50837)
2,node,63253672,52.499322,13.296118,Berlin,DE,39,10711,Joachim-Friedrich-Straße,Halensee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.29612 52.49932)
3,node,253616592,52.511047,13.462698,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.4627 52.51105)
4,node,266629404,52.509209,13.587500,Berlin,DE,45,12621,Mädewalder Weg,Kaulsdorf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.5875 52.50921)



=== DATA QUALITY CHECKS ===
Missing name: 163
Missing address: 777
 Column 'district_id' not found!
Duplicate spaeti_id: 0
Invalid coordinates: 0
 Column 'neighborhood_id' not found!
Missing phone_number: 1538
Missing website: 1551
Missing opening_hours: 888

🔎 Preview of final export (10 rows):


,spaeti_id,name,address,phone_number,email,website,latitude,longitude,opening_hours,beverage_emphasis,has_outdoor_area,geometry
0,26867411,Bavaria petrol,Heilbronner Straße,NaN,NaN,NaN,52.501974,13.294496,Mo-Fr 07:00-22:00; Sa 08:00-22:00,False,False,POINT (13.2945 52.50197)
1,29997723,Aral,Messedamm,NaN,NaN,https://tankstelle.aral.de/tankstelle/berlin/m...,52.508370,13.280947,24/7,False,False,POINT (13.28095 52.50837)
2,63253672,Späti Joe,Joachim-Friedrich-Straße,NaN,NaN,NaN,52.499322,13.296118,24/7,False,False,POINT (13.29612 52.49932)
3,253616592,Mein Markt Pham,NaN,NaN,NaN,NaN,52.511047,13.462698,Mo-Fr 08:00-20:00; Sa 08:00-19:00,False,False,POINT (13.4627 52.51105)
4,266629404,...nah und gut,Mädewalder Weg,+49 30 5677706,e-markt-vollack@t-online.de,NaN,52.509209,13.587500,Mo-Fr 07:00-19:00; Sa 07:00-13:00; PH off,False,False,POINT (13.5875 52.50921)
5,268424318,NaN,NaN,NaN,NaN,NaN,52.596622,13.290950,NaN,False,False,POINT (13.29095 52.59662)
6,269475044,Milchladen,Dresdener Straße,NaN,NaN,NaN,52.501786,13.416537,Mo-Fr 11:00-18:00; Su 13:00-18:00,False,False,POINT (13.41654 52.50179)
7,270719151,Zeitungskiosk Bagci,Dillgesstraße,NaN,NaN,NaN,52.434623,13.342479,NaN,False,False,POINT (13.34248 52.43462)
8,271419848,Berlin Tag und Nacht 2,NaN,NaN,NaN,NaN,52.528971,13.315925,NaN,False,False,POINT (13.31592 52.52897)
9,277681454,Bagdad Lebensmittel,NaN,NaN,NaN,NaN,52.499094,13.393187,NaN,False,False,POINT (13.39319 52.49909)



📐 DataFrame info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1608 entries, 0 to 1607
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   spaeti_id          1608 non-null   int64   
 1   name               1445 non-null   object  
 2   address            831 non-null    object  
 3   phone_number       70 non-null     object  
 4   email              15 non-null     object  
 5   website            57 non-null     object  
 6   latitude           1608 non-null   float64 
 7   longitude          1608 non-null   float64 
 8   opening_hours      720 non-null    object  
 9   beverage_emphasis  1608 non-null   bool    
 10  has_outdoor_area   1608 non-null   bool    
 11  geometry           1608 non-null   geometry
dtypes: bool(2), float64(2), geometry(1), int64(1), object(6)
memory usage: 128.9+ KB

📊 Columns in final export:
['spaeti_id', 'name', 'address', 'phone_number', 'email', 'website

In [16]:
# --- Database connection settings ---
user_name = 'harrison_nwachukwu'
password = 'hk5fxVEI0ReFx5r83'
host = 'localhost'
port = '5433'
database = 'layereddb'
schema = 'berlin_source_data'

# --- Imports ---
import os
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
from sqlalchemy.exc import IntegrityError, SQLAlchemyError

# --- Connect to the database using psycopg2 ---
conn = psycopg2.connect(
    host=host,
    port=port,
    dbname=database,
    user=user_name,
    password=password
)
conn.autocommit = True
cur = conn.cursor()

print("Connected successfully (psycopg2).")

# --- Drop table if exists ---
cur.execute(f"DROP TABLE IF EXISTS {schema}.spaetis CASCADE;")
print("Dropped table spaetis (if existed).")

# --- Create table with FOREIGN KEY constraint ---
create_sql = f"""
CREATE TABLE {schema}.spaetis (
    spaeti_id VARCHAR(20) PRIMARY KEY,
    district_id VARCHAR(8) NOT NULL,
    neighborhood_id VARCHAR(8),
    name VARCHAR(200) NOT NULL,
    type VARCHAR(100),
    address VARCHAR(200) NOT NULL,
    phone_number VARCHAR(50),
    email VARCHAR(100),
    website VARCHAR(200),
    latitude DECIMAL(9,6),
    longitude DECIMAL(9,6),
    opening_hours VARCHAR(200),
    beverage_emphasis BOOLEAN,
    has_outdoor_area BOOLEAN,
    district_name VARCHAR(100),
    neighborhood_name VARCHAR(100),
    coordinates VARCHAR,
    geometry VARCHAR,
    CONSTRAINT district_id_fk FOREIGN KEY (district_id)
        REFERENCES {schema}.districts(district_id)
        ON DELETE RESTRICT ON UPDATE CASCADE
);
"""
cur.execute(create_sql)
print("Created new spaetis table with district_id_fk foreign key.")

# --- Closed psycopg2 connection 
cur.close()
conn.close()
print("Closed psycopg2 connection.")

# Loading spaetis_sql_final.csv
csv_path = "/Users/harrisongoodman/Downloads/spaetis_sql_final.csv"
if not os.path.isfile(csv_path):
    raise FileNotFoundError(f"CSV not found at: {csv_path}")

df_export = pd.read_csv(csv_path)
print("Loaded CSV. Preview:")
print(df_export.head())
print(df_export.info())

# Optional: ensure expected columns exist and handle geometry/coordinates naming
# If geometry exists but you want coordinates string column to be present for the table:
if "coordinates" not in df_export.columns and "geometry" in df_export.columns:
    df_export["coordinates"] = df_export["geometry"].astype(str)
# If geometry isn't desired in the DB insert (but table has 'geometry' as VARCHAR), it's fine to insert it as text.
# If you want to drop geometry before inserting, uncomment the next line:
# df_export = df_export.drop(columns=["geometry"], errors="ignore")

# --- Insert into the database using SQLAlchemy 
engine = create_engine(
    f'postgresql+psycopg2://{user_name}:{password}@{host}:{port}/{database}'
)

try:
    df_export.to_sql(
        "spaetis",
        engine,
        schema=schema,
        if_exists="append",
        index=False
    )
    print("✅ Data successfully inserted into berlin_source_data.spaetis!")
except IntegrityError as ie:
    # Likely a FK violation or primary key conflict
    print("❌ IntegrityError while inserting data (possible FK or PK violation).")
    print(str(ie))
    raise
except SQLAlchemyError as sae:
    print("❌ SQLAlchemy error during insert:")
    print(str(sae))
    raise


Connected successfully (psycopg2).
Dropped table spaetis (if existed).
Created new spaetis table with district_id_fk foreign key.
Closed psycopg2 connection.
Loaded CSV. Preview:
   spaeti_id  district_id  neighborhood_id            name  \
0   26867411     11004004              407  Bavaria petrol   
1   29997723     11004004              405            Aral   
2   63253672     11004004              407       Späti Joe   
3  266629404     11010010             1003  ...nah und gut   
4  269475044     11002002              202      Milchladen   

                    address    phone_number                        email  \
0        Heilbronner Straße             NaN                          NaN   
1                 Messedamm             NaN                          NaN   
2  Joachim-Friedrich-Straße             NaN                          NaN   
3            Mädewalder Weg  +49 30 5677706  e-markt-vollack@t-online.de   
4          Dresdener Straße             NaN                         

In [11]:
import pandas as pd
import os

# Use your existing dataframe with spaeti_id
df = spatis   # or spaetis_final, whichever contains your cleaned data

# 1️⃣ Remove duplicates
df = df.drop_duplicates(subset=["spaeti_id"])

# 2️⃣ Create "coordinates" from geometry only if geometry exists
if "geometry" in df.columns:
    df["coordinates"] = df["geometry"].astype(str)
else:
    df["coordinates"] = pd.NA

# 3️⃣ Required SQL column order
sql_columns = [
    "spaeti_id", "district_id", "neighborhood_id", "name", "type",
    "address", "postal_code", "phone_number", "email", "website",
    "coordinates", "latitude", "longitude", "opening_hours",
    "beverage_emphasis", "has_outdoor_area",
    "district_name", "neighborhood_name", "geometry"
]

# Keep only columns that exist in df
sql_columns_existing = [c for c in sql_columns if c in df.columns]

# 4️⃣ Create final export dataframe
df_export = df[sql_columns_existing]

# 5️⃣ Show summary before insertion/export
print("🔎 Preview of final export (10 rows):")
display(df_export.head(10))

print("\n📐 DataFrame info:")
print(df_export.info())

print("\n📊 Columns in final export:")
print(list(df_export.columns))


🔎 Preview of final export (10 rows):


,spaeti_id,name,address,phone_number,email,website,coordinates,latitude,longitude,opening_hours,geometry
0,26867411,Bavaria petrol,Heilbronner Straße,NaN,NaN,NaN,POINT (13.294496 52.501974),52.501974,13.294496,Mo-Fr 07:00-22:00; Sa 08:00-22:00,POINT (13.2945 52.50197)
1,29997723,Aral,Messedamm,NaN,NaN,https://tankstelle.aral.de/tankstelle/berlin/m...,POINT (13.280947 52.50837),52.508370,13.280947,24/7,POINT (13.28095 52.50837)
2,63253672,Späti Joe,Joachim-Friedrich-Straße,NaN,NaN,NaN,POINT (13.296118 52.499322),52.499322,13.296118,24/7,POINT (13.29612 52.49932)
3,253616592,Mein Markt Pham,NaN,NaN,NaN,NaN,POINT (13.462698 52.511047),52.511047,13.462698,Mo-Fr 08:00-20:00; Sa 08:00-19:00,POINT (13.4627 52.51105)
4,266629404,...nah und gut,Mädewalder Weg,+49 30 5677706,e-markt-vollack@t-online.de,NaN,POINT (13.5875 52.509209),52.509209,13.587500,Mo-Fr 07:00-19:00; Sa 07:00-13:00; PH off,POINT (13.5875 52.50921)
5,268424318,NaN,NaN,NaN,NaN,NaN,POINT (13.29095 52.596622),52.596622,13.290950,NaN,POINT (13.29095 52.59662)
6,269475044,Milchladen,Dresdener Straße,NaN,NaN,NaN,POINT (13.416537 52.501786),52.501786,13.416537,Mo-Fr 11:00-18:00; Su 13:00-18:00,POINT (13.41654 52.50179)
7,270719151,Zeitungskiosk Bagci,Dillgesstraße,NaN,NaN,NaN,POINT (13.342479 52.434623),52.434623,13.342479,NaN,POINT (13.34248 52.43462)
8,271419848,Berlin Tag und Nacht 2,NaN,NaN,NaN,NaN,POINT (13.315925 52.528971),52.528971,13.315925,NaN,POINT (13.31592 52.52897)
9,277681454,Bagdad Lebensmittel,NaN,NaN,NaN,NaN,POINT (13.393187 52.499094),52.499094,13.393187,NaN,POINT (13.39319 52.49909)



📐 DataFrame info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1608 entries, 0 to 1607
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   spaeti_id      1608 non-null   int64   
 1   name           1445 non-null   object  
 2   address        831 non-null    object  
 3   phone_number   70 non-null     object  
 4   email          15 non-null     object  
 5   website        57 non-null     object  
 6   coordinates    1608 non-null   object  
 7   latitude       1608 non-null   float64 
 8   longitude      1608 non-null   float64 
 9   opening_hours  720 non-null    object  
 10  geometry       1608 non-null   geometry
dtypes: float64(2), geometry(1), int64(1), object(7)
memory usage: 138.3+ KB
None

📊 Columns in final export:
['spaeti_id', 'name', 'address', 'phone_number', 'email', 'website', 'coordinates', 'latitude', 'longitude', 'opening_hours', 'geometry']
